<a href="https://colab.research.google.com/github/batinylmz/financialAnomalyDetection/blob/yunusMaster/PCAfinance.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [5]:
import pandas as pd
import numpy as np
from sklearn.decomposition import KernelPCA
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import classification_report, confusion_matrix

# 1. Veriyi Yükle
df = pd.read_csv('/content/drive/MyDrive/financial_anomaly_benchmark_data.csv')

# ==========================================
# ADIM 1: SENTETİK ETİKET (GROUND TRUTH) OLUŞTURMA
# ==========================================
# Sizin Kuralınız: Hacim Değişimi * Volatilite (Bu bizim sınavımızın gizli cevap anahtarı)
df['Anomaly_Score_Synthetic'] = np.abs(df['Volatility_HighLow'] * df['Volume_Change'])

# En yüksek %1'lik kısmı "Gerçek Anomali" kabul ediyoruz.
threshold_true = np.percentile(df['Anomaly_Score_Synthetic'], 99)
df['True_Anomaly'] = (df['Anomaly_Score_Synthetic'] > threshold_true).astype(int)

# ==========================================
# ADIM 2: BİLGİSAYAR ÇÖKMESİNİ ÖNLEMEK İÇİN ÖRNEKLEME
# ==========================================
df_sample = df.sample(n=10000, random_state=42).copy()

# ==========================================
# ADIM 3: EĞİTİM İÇİN YENİ (KOPYASIZ) VERİ HAZIRLIĞI
# ==========================================
# DİKKAT: 'Volatility_HighLow' ve 'Volume_Change' ÇIKARILDI!
# Modelin "kopya" çekmemesi için piyasanın başka dinamiklerini veriyoruz.
# Örneğin:
# 1. Returns (Kapanışlar arası getiri)
# 2. Volume (Hacim değişimini değil, direkt hacmin kendisini veriyoruz)
# 3. Gün İçi Değişim (Kapanış ile Açılış arasındaki fark) -> Kendi ürettiğimiz yeni bir özellik

df_sample['Intraday_Return'] = (df_sample['Close'] - df_sample['Open']) / df_sample['Open']

# Sadece bu 3 yeni/bağımsız özelliği eğitime alıyoruz
features = ['Returns', 'Volume', 'Intraday_Return']
X = df_sample[features]

# Standartlaştırma
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

# ==========================================
# ADIM 4: KERNEL PCA MODELİNİN EĞİTİLMESİ
# ==========================================
print("Kernel PCA (Kopyasız Veri İle) eğitiliyor, lütfen bekleyin...")
kpca = KernelPCA(n_components=2, kernel='rbf', gamma=0.1, fit_inverse_transform=True, random_state=42)

# Sıkıştırma
X_kpca = kpca.fit_transform(X_scaled)

# Geri Açma
X_reconstructed = kpca.inverse_transform(X_kpca)

# ==========================================
# ADIM 5: ANOMALİ TESPİTİ VE KIYASLAMA
# ==========================================
reconstruction_error = np.mean(np.square(X_scaled - X_reconstructed), axis=1)
df_sample['Reconstruction_Error'] = reconstruction_error

threshold_pred = np.percentile(reconstruction_error, 98)
df_sample['Predicted_Anomaly'] = (reconstruction_error > threshold_pred).astype(int)

# ==========================================
# ADIM 6: SONUÇLARI GÖRÜNTÜLEME
# ==========================================
print("\n--- MODEL BAŞARISI: HATA MATRİSİ (CONFUSION MATRIX) ---")
print(confusion_matrix(df_sample['True_Anomaly'], df_sample['Predicted_Anomaly']))

print("\n--- DETAYLI SINIFLANDIRMA RAPORU ---")
print(classification_report(df_sample['True_Anomaly'], df_sample['Predicted_Anomaly']))

Kernel PCA (Kopyasız Veri İle) eğitiliyor, lütfen bekleyin...

--- MODEL BAŞARISI: HATA MATRİSİ (CONFUSION MATRIX) ---
[[9761  147]
 [  39   53]]

--- DETAYLI SINIFLANDIRMA RAPORU ---
              precision    recall  f1-score   support

           0       1.00      0.99      0.99      9908
           1       0.27      0.58      0.36        92

    accuracy                           0.98     10000
   macro avg       0.63      0.78      0.68     10000
weighted avg       0.99      0.98      0.98     10000

